# Stimulus Feature Extraction

This notebook demonstrates the `pyeeg.features` module for extracting
linguistic and acoustic features from naturalistic stimuli for TRF
analysis.

## Overview

The `StimulusEncoder` provides a unified interface for:
- **LLM features**: surprisal, entropy, KL divergence from GPT-2 models
- **Syntactic features**: depth, opening, closing from constituency parses
- **Acoustic features**: envelope, filterbank, gammatone rate maps
- **User-defined extractors**: wrap any callable as a feature extractor

Features can be:
- Composed via `FeaturePipeline`
- Aligned to neural signal samples via `AlignmentHandler` + TextGrid
- Normalized (zscore, minmax)
- Reduced (PCA, ICA) via `FeatureReducer`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyeeg.features import (
    LLMFeatureExtractor, LLMFeatureConfig,
    StimulusEncoder,
    FeaturePipeline, PipelineConfig, FeatureSpec,
    AlignmentHandler, TextGridParser,
    FeatureReducer, ReductionConfig,
)
from pyeeg.features.acoustic import AcousticFeatureExtractor, AcousticFeatureConfig

## 1. LLM Feature Extraction

Extract surprisal, entropy, and KL divergence from text using a GPT-2
language model. We use a locally-built tiny GPT-2 for this demo (no
network required). For real analysis, use `distilgpt2` or
`GroNLP/gpt2-small-dutch`.

In [ ]:
import os

# Use the tiny local model (run tests/build_test_model.py to create it)
model_path = os.path.expanduser('~/.cache/huggingface/tiny-gpt2-test')

if not os.path.exists(model_path):
    print('Building tiny GPT-2 model...')
    import subprocess, sys
    subprocess.run([sys.executable, 'tests/build_test_model.py'])

config = LLMFeatureConfig(model_name=model_path, device='cpu')
extractor = LLMFeatureExtractor(config)

text = 'The cat sat on the mat. The dog ran in the park.'
features = extractor.extract(text, return_word_level=True)

print(f'Text: {text!r}')
print(f'Words: {len(features["surprisal"])}')
print()
for name, values in features.items():
    print(f'{name:20s}: {values}')

In [ ]:
# Visualize surprisal across words
words = text.split()
surprisal = features['surprisal']

fig, ax = plt.subplots(figsize=(10, 3))
x = np.arange(len(surprisal))
ax.bar(x, np.nan_to_num(surprisal, nan=0), color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_ylabel('Surprisal (nats)')
ax.set_title('Word-level Surprisal')
plt.tight_layout()
plt.show()

## 2. Using StimulusEncoder for Pipeline

The `StimulusEncoder` composes multiple extractors and handles alignment
and normalization.

In [ ]:
# Create an encoder with LLM features
encoder = StimulusEncoder()
encoder.add_llm_features(
    features=['surprisal', 'entropy', 'kl_divergence'],
    model_name=model_path
)

text = 'The cat sat on the mat.'
features, metadata = encoder.encode(text)

print('Features:')
for name, values in features.items():
    print(f'  {name}: {values}')
print(f'\nMetadata: {metadata}')

## 3. Alignment to Neural Signal

Align word-level features to neural signal samples using a Praat TextGrid.

In [ ]:
# Create a TextGrid with word intervals
textgrid_string = '''"""
File type = "ooTextFile"
Object class = "TextGrid"

xmin = 0
xmax = 3
tiers? <exists>
size = 1
item []:
    item [1]:
        class = "IntervalTier"
        name = "words"
        xmin = 0
        xmax = 3
        intervals: size = 3
        intervals [1]:
            xmin = 0
            xmax = 1
            text = "The"
        intervals [2]:
            xmin = 1
            xmax = 2
            text = "cat"
        intervals [3]:
            xmin = 2
            xmax = 3
            text = "sat"
"""

handler = AlignmentHandler(signal_sampling_rate=1000.0)
textgrid = handler.load_textgrid_from_string(textgrid_string)

# Set up aligned encoder
encoder = StimulusEncoder()
encoder.add_llm_features(features=['surprisal'], model_name=model_path)
encoder.set_alignment(sampling_rate=1000.0)

features, metadata = encoder.encode(
    'The cat sat', textgrid=textgrid, signal_length=3000
)

print(f'Aligned shape: {features["llm_surprisal"].shape}')
print(f'Metadata: {metadata}')

# Plot aligned features
fig, ax = plt.subplots(figsize=(10, 3))
t = np.arange(features['llm_surprisal'].shape[0]) / 1000.0
ax.plot(t, features['llm_surprisal'], color='steelblue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Surprisal')
ax.set_title('Aligned Surprisal to Neural Signal Samples')
ax.set_xlim(0, 3)
plt.tight_layout()
plt.show()

## 4. Normalization

Apply z-score or min-max normalization to features.

In [ ]:
from pyeeg.features.pipeline import PipelineConfig

# Z-score normalized encoder
encoder = StimulusEncoder(
    PipelineConfig(normalization='zscore')
)
encoder.add_llm_features(features=['surprisal', 'entropy'], model_name=model_path)

text = 'The cat sat on the mat. The dog ran in the park.'
features_norm, _ = encoder.encode(text)

print('Z-scored features:')
for name, values in features_norm.items():
    valid = values[~np.isnan(values)]
    print(f'  {name}: mean={valid.mean():.4f}, std={valid.std():.4f}')

## 5. Acoustic Feature Extraction

Extract envelope, filterbank, and gammatone features from audio signals.

In [ ]:
# Create a synthetic audio signal (1 second, 440 Hz + 2000 Hz)
srate = 16000
t = np.arange(srate) / srate
audio = 0.5 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 2000 * t)

# Extract envelope
acoustic_ext = AcousticFeatureExtractor(
    AcousticFeatureConfig(features=['envelope'], sampling_rate=srate)
)
envelope = acoustic_ext.extract(audio, srate)

print(f'Envelope shape: {envelope["envelope"].shape}')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t[:1000], audio[:1000], color='gray', alpha=0.7)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Audio Signal (first 62.5 ms)')

env_t = np.arange(len(envelope['envelope'])) / 125.0  # envelope at 125 Hz
axes[1].plot(env_t, envelope['envelope'], color='steelblue', linewidth=2)
axes[1].set_ylabel('Envelope')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Acoustic Envelope')
plt.tight_layout()
plt.show()

In [ ]:
# Acoustic features via StimulusEncoder
encoder = StimulusEncoder()
encoder.add_acoustic_features(
    features=['envelope'], sampling_rate=srate
)

features, meta = encoder.encode_audio(audio, srate)
print(f'Acoustic features: {list(features.keys())}')
print(f'Envelope shape: {features["acoustic_envelope"].shape}')
print(f'Metadata: {meta}')

## 6. User-Defined Extractors

Wrap any callable as a feature extractor.

In [ ]:
# Define a custom feature: word frequency (simplified)
word_freq = {
    'the': 0.9, 'cat': 0.1, 'sat': 0.2, 'on': 0.8,
    'mat': 0.05, 'dog': 0.1, 'ran': 0.2, 'in': 0.7,
    'park': 0.05
}

def word_frequency_extractor(text):
    words = text.lower().rstrip('.').split()
    return {'frequency': np.array([word_freq.get(w, 0.01) for w in words])}

# Add to encoder alongside LLM features
encoder = StimulusEncoder()
encoder.add_llm_features(features=['surprisal'], model_name=model_path)
encoder.add_custom_extractor(word_frequency_extractor, name='wordfreq')

text = 'The cat sat on the mat.'
features, meta = encoder.encode(text)

print('LLM + custom features:')
for name, values in features.items():
    print(f'  {name}: {values}')

## 7. Dimensionality Reduction

Reduce high-dimensional features (e.g. filterbank) with PCA or ICA.

In [ ]:
# Extract filterbank features
acoustic_ext = AcousticFeatureExtractor(
    AcousticFeatureConfig(features=['filterbank'], sampling_rate=srate)
)
fb = acoustic_ext.extract(audio, srate)
fb_data = fb['filterbank']  # (n_filters, n_samples)

# Transpose to (n_samples, n_filters) for the reducer
fb_2d = fb_data.T
print(f'Filterbank shape: {fb_2d.shape}')

# Reduce with PCA to 3 components
reducer = FeatureReducer(ReductionConfig(method='pca', n_components=3))
reduced = reducer.fit_transform(fb_2d)
print(f'Reduced shape: {reduced.shape}')
print(f'Explained variance: {reducer.get_explained_variance()}')

# Plot reduced components
fig, ax = plt.subplots(figsize=(10, 3))
for i in range(3):
    ax.plot(t, reduced[:, i], label=f'PC{i+1}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('PCA-Reduced Filterbank Components')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. **LLM features**: surprisal, entropy, KL divergence from GPT-2
2. **Pipeline**: `StimulusEncoder` composes extractors
3. **Alignment**: word features aligned to signal samples via TextGrid
4. **Normalization**: z-score and min-max
5. **Acoustic features**: envelope, filterbank, gammatone
6. **Custom extractors**: wrap any callable
7. **Dimensionality reduction**: PCA, ICA via `FeatureReducer`

For real experiments, use a pre-trained model like `distilgpt2` or
`GroNLP/gpt2-small-dutch`. Download distilgpt2 with:
```bash
uv run --extra features python tests/download_distilgpt2.py
```